# NPC converter playground

An interactive, exploratory front end for analytical and local numerical `JetCocoon`/NPC input models. It is deliberately separate from the audited export workflow.

From the repository root, install the optional notebook dependencies and start JupyterLab:

```bash
python -m pip install -e '.[notebooks]'
jupyter lab notebooks/np_converter_playground.ipynb
```

The cells below are unexecuted on purpose: edit parameters, then run them in order.

In [ ]:
from __future__ import annotations

import importlib
import os
import sys
from dataclasses import replace
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
from IPython.display import FileLink, Image, display

# This works when opened from the repository root or from notebooks/.
_start = Path.cwd().resolve()
REPO_ROOT = None
for candidate in (_start, *_start.parents):
    if (candidate / 'examples').is_dir() and (candidate / 'src').is_dir():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook inside the jetbns repository.')
EXAMPLES = REPO_ROOT / 'examples'
if str(EXAMPLES) not in sys.path:
    sys.path.insert(0, str(EXAMPLES))

import npc_playground as _npc_playground  # noqa: E402
from export_npc_models import ANALYTICAL, NUMERICAL  # noqa: E402

importlib.reload(_npc_playground)
from npc_playground import (  # noqa: E402
    launch_dashboard,
    plot_comparisons,
    plot_run,
    run_model,
    save_run,
    summary,
    table,
)
from plot_npc_model_comparison import plot_comparison  # noqa: E402

from jetbns import NpcConfig  # noqa: E402

# Set this to the directory containing any private local numerical profiles.
# This checkout keeps the legacy trees below codes/.archived.
DATA_ROOT = REPO_ROOT.parent
archived_data_root = DATA_ROOT / '.archived'
if any((archived_data_root / model.profile).is_file() for model in NUMERICAL):
    DATA_ROOT = archived_data_root
print(f'Numerical-profile root: {DATA_ROOT}')


## Editable quick-run parameters

These settings prioritize iteration time. They are **not** a convergence
certificate; use the formal exporter for reproducible, audited products.
Engine luminosity is isotropic-equivalent in `Model`, while the helper
converts it to the true one-sided top-hat jet power.

In [ ]:
# Engine controls
ENGINE_GAMMA = 100.0
OPENING_ANGLE_DEG = 10.0
TIME_STEP_S = 0.01
MAX_DURATION_S = 15.0
SAMPLE_COUNT = 201

# Numerical-ejecta reconstruction controls (used only by numerical models).
KERNEL_SHAPE = 2.0
BETA_WIDTH = 0.05
CUTOFF_MODE = 'smooth'

# Ejecta / engine controls exposed through Model.
EJECTA_MASS_MSUN = 1e-4
TAIL_EXPONENT = 8.0
ELECTRON_FRACTION = 0.30
LAUNCH_S = 1.0
LUMINOSITY_ISO_ERG_S = 1e51

# Keep path length and composition choices explicit in exploratory runs.
NPC_CONFIG = NpcConfig(electron_fraction=ELECTRON_FRACTION, path_length='radius')

reference = replace(
    ANALYTICAL[0], name='playground_reference', mass_msun=EJECTA_MASS_MSUN,
    tail_exponent=TAIL_EXPONENT, electron_fraction=ELECTRON_FRACTION,
    launch_s=LAUNCH_S, luminosity_iso_erg_s=LUMINOSITY_ISO_ERG_S,
)
# A useful contrast: lower Ye and a more energetic, steeper tail.
neutron_rich = replace(
    ANALYTICAL[1], name='playground_neutron_rich', electron_fraction=0.10,
    mass_msun=2e-4, tail_exponent=12.0, launch_s=LAUNCH_S,
    luminosity_iso_erg_s=3e51,
)
MODELS = (reference, neutron_rich)


In [ ]:
runs = []
for model in MODELS:
    run = run_model(
        model, DATA_ROOT, lorentz_factor=ENGINE_GAMMA,
        opening_angle_deg=OPENING_ANGLE_DEG, time_step_s=TIME_STEP_S,
        max_duration_s=MAX_DURATION_S, sample_count=SAMPLE_COUNT,
        kernel_shape=KERNEL_SHAPE, beta_width=BETA_WIDTH, cutoff_mode=CUTOFF_MODE,
        npc_config=replace(NPC_CONFIG, electron_fraction=model.electron_fraction),
    )
    runs.append(run)
    print(f'Completed {model.name}')
display(summary(runs))


In [ ]:
# Ejecta, head propagation, and cocoon diagnostics for one selected run.
first = runs[0]
fig = plot_run(first)
display(fig)
plt.close(fig)


In [ ]:
# The four existing 31-panel NPC comparison pages.
comparison_figures = plot_comparisons(runs, normalized=True, near_breakout=False)
for figure in comparison_figures:
    display(figure)
    plt.close(figure)

# All sampled NPC arrays for the selected run, ready for local inspection.
display(table(first))


## Optional local numerical profile

Numerical profiles are private/local inputs and are not supplied by this
repository. Set `RUN_NUMERICAL=True` only after `DATA_ROOT` points at a
tree containing the selected profile. Numerical reconstruction can be
substantially slower.

In [ ]:
RUN_NUMERICAL = False
numerical_run = None
if RUN_NUMERICAL:
    numerical_model = NUMERICAL[0]
    profile_path = DATA_ROOT / numerical_model.profile
    if not profile_path.is_file():
        print(f'Missing local numerical profile: {profile_path}')
        print('Set DATA_ROOT to the profile-data root, or leave '
              'RUN_NUMERICAL=False.')
    else:
        numerical_run = run_model(
            numerical_model, DATA_ROOT, lorentz_factor=ENGINE_GAMMA,
            opening_angle_deg=OPENING_ANGLE_DEG, time_step_s=TIME_STEP_S,
            max_duration_s=MAX_DURATION_S, sample_count=SAMPLE_COUNT,
            kernel_shape=KERNEL_SHAPE, beta_width=BETA_WIDTH,
            cutoff_mode=CUTOFF_MODE,
            npc_config=NPC_CONFIG,
        )
        display(summary([numerical_run]))
        numerical_figure = plot_run(numerical_run)
        display(numerical_figure)
        plt.close(numerical_figure)


## Preview an existing six-model export (no solver rerun)

When a prior formal export is present, this optional cell lists the actual
model groups in its HDF5 file, regenerates comparison images from its stored
tables, and links to the PDF. It never launches the propagation solver.

In [10]:
LOAD_EXISTING = False
EXISTING_EXPORT = (
    REPO_ROOT / 'examples' / 'output' / 'npc_model_export'
    / 'npc_prebreakout_models.h5'
)
EXISTING_COMPARISON_DIR = EXISTING_EXPORT.parent / 'comparison'
if LOAD_EXISTING:
    if not EXISTING_EXPORT.is_file():
        print(f'Missing existing export: {EXISTING_EXPORT}')
        print('Run the formal exporter first, or set EXISTING_EXPORT '
              'to its HDF5 path.')
    else:
        with h5py.File(EXISTING_EXPORT) as handle:
            model_names = list(handle['prebreakout_summary/name'].asstr()[:])
            print(f"Schema: {handle.attrs.get('schema', 'unknown')}")
            print(f"Stored models ({len(model_names)}):")
            print(', '.join(model_names))
        preview_paths = plot_comparison(EXISTING_EXPORT, EXISTING_COMPARISON_DIR)
        png_paths = [path for path in preview_paths if path.suffix == '.png']
        for png_path in png_paths:
            display(Image(filename=str(png_path)))
        pdf_path = next(path for path in preview_paths if path.suffix == '.pdf')
        relative_pdf_path = Path(os.path.relpath(pdf_path, Path.cwd()))
        display(FileLink(
            str(relative_pdf_path), result_html_prefix='Comparison PDF: '
        ))


## Optional local export

This produces convenient single-run artifacts, not the distributable audited
archive. For the reviewed HDF5/PDF/ZIP suite, use
`python examples/export_npc_models.py` and its documented options.

In [11]:
SAVE_OUTPUT = False
OUTPUT_DIR = REPO_ROOT / 'examples' / 'output' / 'playground'
if SAVE_OUTPUT:
    paths = save_run(first, OUTPUT_DIR)
    display(paths)


## Interactive dashboard

Press **Run and add model** for the first solve. After that, changing a slider
reruns and replaces the latest result when **Update after slider release** is
enabled. Give a trial a new name and press the button to retain it as a separate
comparison model. The dashboard does not stream ODE integration or expose an
ODE progress hook. Long numerical-profile solves can be interrupted with
**Kernel → Interrupt**. Normalized time is `(t - t0) / (tbo - t0)` only for
trajectories that break out.

Scientific cautions: the free-proton quantity is a composition proxy, the
free-neutron skin is a phenomenological Metzger et al. estimate rather than
a reaction-network result, and local optical-depth estimates are distinct
from radial columns. Quick runs are not convergence certified.

In [12]:
dashboard = launch_dashboard(DATA_ROOT)
display(dashboard)
